# Demo 5 — Phát hiện shift: thống kê, test và cửa sổ thời gian

**Slide:** 14 và 15 · **Thời lượng:** 60–90 giây · **Hình:** `05_detect_shift.png`

## Ý chính

1. **Summary statistics** cho tín hiệu nhanh nhưng không đủ.
2. **Two-sample test** (KS cho 1 chiều, MMD cho đa biến) kiểm định “có khác phân phối không”.
3. **Cửa sổ thời gian** quyết định kết luận: cùng dữ liệu, cửa sổ sai → false alarm.
4. **Sliding vs cumulative**: cumulative che mất sự cố ngắn hạn.

> **Câu chốt:** phải biết loại nào, đo bằng gì, trong cửa sổ nào.

## Bước 0 — Thiết lập

In [ ]:
# --- Thiết lập: tìm và import helper từ ../code-demo ---
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

for candidate in [
    Path.cwd() / "code-demo",
    Path.cwd().parent / "code-demo",
    Path.cwd().parent.parent / "code-demo",
]:
    if (candidate / "common.py").exists():
        CODE_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError("Không tìm thấy code-demo/common.py")

sys.path.insert(0, str(CODE_DIR))

from common import (
    ks_perm_test,
    mmd_perm_test,
    print_stats_table,
    save,
    setup,
)  # noqa: E402

setup()
print("code-demo:", CODE_DIR)
print("Hình sẽ lưu vào:", CODE_DIR / "figures")

## Phần A — Summary statistics & two-sample test

So sánh 3 tập 1500 mẫu:

- `source`: chuẩn tắc $N(0,1)$.
- `same`: cũng $N(0,1)$ — không shift.
- `shifted`: $N(0.6, 1.3)$ — có shift.

KS và MMD đều dùng **permutation p-value**, không cần SciPy.

In [ ]:
rng = np.random.default_rng(21)

source = rng.normal(0.0, 1.0, 1500)
same = rng.normal(0.0, 1.0, 1500)
shifted = rng.normal(0.6, 1.3, 1500)

print("=== DEMO 5A: summary statistics & two-sample test ===")
print("\n[Không shift] source vs target cùng phân phối")
print_stats_table(source, same)

print("\n[Có shift] source vs target khác phân phối")
print_stats_table(source, shifted)

for name, target in [("cùng phân phối", same), ("khác phân phối", shifted)]:
    ks, p_ks = ks_perm_test(source, target, n_perm=1000)
    mmd, p_mmd = mmd_perm_test(source[:, None], target[:, None], n_perm=300)
    print(f"\n{name:>16}: KS D={ks:.4f}, p={p_ks:.4f} | MMD²={mmd:.5f}, p={p_mmd:.4f}")

print("\nLưu ý: p-value nhỏ → bác bỏ 'cùng phân phối'.")
print("Nhưng ý nghĩa thống kê ≠ ý nghĩa thực tiễn: mẫu càng lớn, test càng nhạy.")

## Phần B — Cửa sổ thời gian

Dữ liệu có chu kỳ tuần. Ngày cuối là **thứ 2 bình thường**.

- Cửa sổ **bỏ sót thứ 2** (Tue–Sun) → baseline lệch → báo shift sai.
- Cửa sổ **đủ 12 tuần** → không báo shift.

Cùng dữ liệu, chỉ đổi cửa sổ baseline → kết luận trái ngược.

In [ ]:
weekly = np.array([1.5, 0.0, -0.5, -1.0, -0.8, -0.3, 0.5])   # Thứ 2 → CN
n_weeks = 12
n_days = n_weeks * 7 + 1
days = np.arange(1, n_days + 1)
weekday = (days - 1) % 7

daily_mean = weekly[weekday] + rng.normal(0, 0.15, n_days)
daily_mean[-1] = weekly[0]   # ngày cuối là thứ 2 "bình thường"

short_mask = (days < n_days) & (weekday != 0)   # cửa sổ bỏ sót thứ 2
long_mask = days < n_days                       # đủ 12 tuần
last = np.array([daily_mean[-1]])

_, p_short = ks_perm_test(daily_mean[short_mask], last, n_perm=500)
_, p_long = ks_perm_test(daily_mean[long_mask], last, n_perm=500)

print("=== DEMO 5B: cửa sổ thời gian ===")
print(f"Cửa sổ bỏ sót thứ 2 (Tue–Sun) vs ngày cuối: KS p = {p_short:.4f}  → {'BÁO SHIFT' if p_short < 0.05 else 'không shift'}")
print(f"Cửa sổ đủ 12 tuần            vs ngày cuối: KS p = {p_long:.4f}  → {'BÁO SHIFT' if p_long < 0.05 else 'không shift'}")
print("Cùng dữ liệu, chỉ đổi cửa sổ baseline → kết luận trái ngược.")
print("Vì dữ liệu có chu kỳ tuần; cửa sổ ngắn bỏ sót chu kỳ → false alarm.")

## Phần C — Sliding vs cumulative accuracy

- **Sliding 4 giờ**: trung bình 4 giờ gần nhất → thấy cú sụt.
- **Cumulative**: trung bình từ đầu → bị pha loãng bởi dữ liệu cũ.

In [ ]:
hours = np.arange(24)
acc_true = np.full(24, 0.90)
acc_true[16:19] = 0.60   # sự cố giờ 16–18
samples_per_hour = 800

observed = np.array([rng.binomial(samples_per_hour, p) / samples_per_hour for p in acc_true])

window = 4
sliding = np.array([observed[max(0, h - window + 1) : h + 1].mean() for h in hours])
cumulative = np.array([observed[: h + 1].mean() for h in hours])

print("=== DEMO 5C: sliding vs cumulative ===")
print(f"Accuracy thật giờ 17          : {acc_true[17]:.2f}")
print(f"Sliding (4h) tại giờ 18       : {sliding[18]:.2f}  ← thấy cú sụt")
print(f"Cumulative tại giờ 18         : {cumulative[18]:.2f}  ← che mất cú sụt")
print("→ Cumulative bị pha loãng bởi dữ liệu cũ.")

## Phần D — Vẽ hình cho slide

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

ax = axes[0]
ax.plot(days, daily_mean, "o-", ms=3, lw=1, color="#1f77b4", label="Trung bình ngày")
ax.scatter(days[-1], daily_mean[-1], s=90, color="#d62728", zorder=5, label="Ngày cuối (thứ 2)")
ax.axhline(weekly[0], color="gray", ls="--", lw=1, label="Mức thứ 2 bình thường")
ax.set_xlabel("Ngày (12 tuần + 1 ngày)")
ax.set_ylabel("Giá trị trung bình")
ax.set_title(f"Cửa sổ thiếu thứ 2: p={p_short:.3f} | đủ chu kỳ: p={p_long:.3f}")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(hours, observed, "o", alpha=0.45, label="accuracy từng giờ", color="gray")
ax.plot(hours, sliding, "-", lw=2, label="sliding 4 giờ", color="#d62728")
ax.plot(hours, cumulative, "--", lw=2, label="cumulative", color="#2ca02c")
ax.axvspan(16, 18, alpha=0.12, color="#d62728")
ax.set_xlabel("Giờ")
ax.set_ylabel("Accuracy")
ax.set_title("Cumulative che mất sự cố ngắn hạn")
ax.legend(fontsize=9)

save(fig, "05_detect_shift.png")
plt.show()

## Đọc kết quả & ý nghĩa cho slide

| Phép đo | Kết quả kỳ vọng | Kết luận |
|---|---|---|
| KS cùng phân phối | p ≈ **0.44** | Không shift |
| MMD cùng phân phối | p ≈ **0.28** | Không shift |
| KS khác phân phối | p ≈ **0.001** | Báo shift |
| MMD khác phân phối | p ≈ **0.003** | Báo shift |
| Cửa sổ bỏ sót thứ 2 | p ≈ **0.026** | **Báo shift sai** |
| Cửa sổ đủ 12 tuần | p ≈ **0.156** | Không shift |
| Sliding 4h tại giờ 18 | **0.67** | Thấy sụt |
| Cumulative tại giờ 18 | **0.85** | Che mất sụt |

**Nói trên slide:**

> “KS và MMD bắt đúng ca khác phân phối, p xấp xỉ 0,001. Nhưng cùng dữ liệu, nếu cửa sổ baseline bỏ sót thứ 2, p = 0,026 — báo shift sai; cửa sổ đủ 12 tuần, p = 0,156 — không shift. Và sliding thấy cú sụt còn 0,67, cumulative chỉ còn 0,85.”

**Câu chốt:** phải biết loại nào, đo bằng gì, trong cửa sổ nào.

**Dùng cho slide nào?**

- Slide 14 — accuracy không đủ; summary statistics.
- Slide 15 — two-sample test & time-scale windows.